# Semantic response caching + durable chat history on Oracle AI Database

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/langchain_ecosystem/langchain_oracle_semantic_cache_chat_history.ipynb)

> **Who this is for:** AI developers who are paying for every LLM call and want two things the database can give them for free: **stop paying to answer the same question twice**, and **never lose a conversation**. Both are one `langchain-oracledb` primitive away.

## The use case: a support assistant that doesn't repeat work

Real users ask the *same thing in different words* all day long — *"How can semantic caching cut LLM costs?"* and *"How do I use a response cache to save on model spend?"* are the same question. A naive app pays Claude to answer both. This notebook wires up two Oracle primitives so it doesn't:

- **`OracleSemanticCache`** — before calling the model, check whether a **meaning-equivalent** question has already been answered. If so, return the stored answer in milliseconds and skip the LLM entirely.
- **`OracleChatMessageHistory`** — persist each session's transcript in Oracle so a later request (or a process restart) recovers the exact conversation.

The subtle, important part — and the thing this notebook proves — is that these two scopes are **independent**: a cached answer can be *reused across different users' sessions*, while each user's *transcript stays perfectly isolated*.

## The stack: PALO + a pluggable vectorizer

This series builds on the **PALO stack** — the *Palo Alto Agentic Stack*: **P**ython · **A**nthropic (Claude) · **L**angChain · **O**racle AI Database. Semantic caching adds one interchangeable part — an **embedding model** to measure "same meaning":

| Layer | This notebook | Role |
|-------|---------------|------|
| **P** — Python | the notebook runtime | glue |
| **A** — Anthropic | `ChatAnthropic` → Claude | the expensive model whose calls we cache |
| **L** — LangChain | `langchain-oracledb`, `langchain-core` | the cache/history primitives + the runnable that wires history in |
| **O** — Oracle | Oracle AI Database 23ai/26ai | stores the cache vectors **and** the chat transcripts |
| *vectorizer* | **OpenAI** `text-embedding-3-small` | turns each question into a vector so "same meaning" becomes "small distance" |

We deliberately use a **real** embedding model here (not an in-database ONNX model, to keep setup to a single `pip install` and one API key). Any LangChain `Embeddings` object would drop in — the Oracle primitives don't care which.

## How a cache-first request flows

Every request follows the same path. The green branch is the whole point: when the incoming question is *semantically close enough* to one already cached, Claude is never called.

```mermaid
flowchart TD
    Q["New user question"] --> EMB["Embed with OpenAI<br/>text-embedding-3-small"]
    EMB --> LOOK{"OracleSemanticCache:<br/>nearest cached question<br/>within score_threshold?"}
    LOOK -->|"HIT (small cosine distance)"| CACHED["Return stored answer<br/>⚡ no LLM call"]
    LOOK -->|"MISS"| LLM["Call Claude<br/>(ChatAnthropic)"]
    LLM --> STORE["Store (question → answer)<br/>in Oracle"]
    STORE --> ANS["Answer"]
    CACHED --> ANS
    ANS --> HIST[("OracleChatMessageHistory:<br/>append turn to this session")]
```

Read it top-to-bottom: embed the question, ask Oracle for the nearest previously-answered question, and either reuse that answer or pay for a fresh Claude call — then record the turn in this session's durable transcript.

## The two Oracle primitives, conceptually

- **`OracleSemanticCache`** is a vector-backed key/value cache. The *key* is the **embedding of the prompt**; a lookup is a nearest-neighbour search under a `distance_strategy` (we use **cosine**). A result counts as a **hit** only if its distance is within `score_threshold`. Entries are also namespaced by an `llm_string` (which model/settings produced them), so the same question under a different model is a separate entry.
- **`OracleChatMessageHistory`** is durable, ordered, `session_id`-scoped message storage. It plugs into LangChain's message-history runnable so history is loaded before a turn and saved after — and `history_size` can bound how many messages you *read back* without deleting anything.

Both are plain LangChain interfaces backed by real Oracle tables — [Oracle AI Vector Search](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/overview-ai-vector-search.html) for the vectors, ordinary relational rows for the transcript.

## The packages, and the job each one does

| Package | What it is | Role here |
|---------|-----------|-----------|
| [`langchain-oracledb`](https://pypi.org/project/langchain-oracledb/) | Oracle's LangChain integration | Provides `OracleSemanticCache` and `OracleChatMessageHistory` (and `DistanceStrategy`). **The star.** |
| [`langchain-openai`](https://docs.langchain.com/oss/python/integrations/text_embedding/openai) | OpenAI models for LangChain | `OpenAIEmbeddings` — turns questions into vectors for the cache. |
| [`langchain-anthropic`](https://docs.langchain.com/oss/python/integrations/chat/anthropic) | Anthropic models for LangChain | `ChatAnthropic` — the Claude call we want to avoid repeating. |
| `langchain-core` | LangChain primitives | `Generation`, `RunnableWithMessageHistory`, message types. |
| [`oracledb`](https://pypi.org/project/oracledb/) | Python driver for Oracle | Opens the (thin-mode, no client install) connection. |

Everything is a released PyPI package — no local, editable, or VCS installs.

---
# Part 1 — Install the packages

One line, latest releases. Re-run any time; restart the kernel if pip replaces an already-imported package.

In [4]:
%pip install -qU \
    langchain-oracledb \
    langchain-openai \
    langchain-anthropic \
    langchain-core \
    oracledb
print("✅ Latest releases installed.")

Note: you may need to restart the kernel to use updated packages.
✅ Latest releases installed.


---
# Part 2 — Start Oracle AI Database

We need one Oracle container — it will hold both the cache vectors and the chat transcripts. Use Gerald Venzl's [`gvenzl/oracle-free`](https://github.com/gvenzl/oci-oracle-free) image (Oracle AI Database Free, vector engine built in). Oracle vector support requires database **23.4+**; we pin `23.26.2`.

```bash
docker run --name oracle-ai-free-23-26-2 \
  --publish 1521:1521 \
  --env ORACLE_PASSWORD=OracleAdminPwd_2026 \
  --env APP_USER=NOTEBOOK_USER \
  --env APP_USER_PASSWORD=NotebookPwd_2026 \
  --health-cmd=healthcheck.sh --health-interval=10s \
  --health-timeout=5s --health-retries=30 \
  --detach gvenzl/oracle-free:23.26.2

# wait until healthy
until [ "$(docker inspect --format='{{.State.Health.Status}}' oracle-ai-free-23-26-2)" = healthy ]; do sleep 5; done
```

The container creates `NOTEBOOK_USER` (password `NotebookPwd_2026`) in service `FREEPDB1` — exactly the defaults the next cell reads. Override `ORACLE_USER` / `ORACLE_PASSWORD` / `ORACLE_DSN` to point at a different database.

---
# Part 3 — Connect and provide keys

Two providers, one database. We read the Anthropic key (for Claude) and the OpenAI key (for embeddings) from the environment — never hard-coded — and open one [`python-oracledb`](https://pypi.org/project/oracledb/) connection in **thin mode** (no Oracle client install required).

In [5]:
import getpass
import os

# Two API keys: Claude answers new questions; OpenAI embeds questions for the cache.
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

ORACLE_USER = os.environ.get("ORACLE_USER", "NOTEBOOK_USER")
ORACLE_PASSWORD = os.environ.get("ORACLE_PASSWORD", "NotebookPwd_2026")
ORACLE_DSN = os.environ.get("ORACLE_DSN", "localhost:1521/FREEPDB1")

In [6]:
import oracledb

connection = oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=ORACLE_DSN)

# Vector features need Oracle 23.4+; thin mode means no native client to install.
db_major_minor = tuple(int(p) for p in connection.version.split(".")[:2])
assert db_major_minor >= (23, 4), f"Need Oracle 23.4+, got {connection.version}"
print(f"✅ Connected as {ORACLE_USER} to {ORACLE_DSN}; database {connection.version}; thin mode.")

✅ Connected as NOTEBOOK_USER to localhost:1521/FREEPDB1; database 23.26.0.0.0; thin mode.


---
# Part 4 — The embedding model, and calibrating "same meaning"

The cache decides *hit vs miss* by embedding the incoming question and measuring its **cosine distance** to previously-cached questions. So the embedding model is the heart of the system. We use OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings) via [`OpenAIEmbeddings`](https://docs.langchain.com/oss/python/integrations/text_embedding/openai) — 1536 dimensions, cheap, and strong at paraphrase similarity.

A crucial, often-skipped point: **distance ranges are model-specific**, so you must calibrate `score_threshold` to *your* embedding model. The next cell measures three real distances — a paraphrase and an unrelated question against a base question — so we can pick a threshold that accepts the paraphrase and rejects the unrelated one.

### Distance vs. similarity (relevance) — the same number, flipped

There are two ways to score how alike two vectors are, and they're easy to conflate:

- **Cosine *distance*** — what Oracle's `VECTOR_DISTANCE(..., COSINE)` and the cache's `score_threshold` actually use. **Smaller is closer**: `0.0` = identical meaning, larger = more different. You rank matches by distance **ascending**.
- **Cosine *similarity* / relevance** — the intuitive "how relevant, 0→1" score you'd show a user. **Larger is closer**: `1.0` = identical. You rank matches **descending**.

They're the same information flipped: **`relevance = 1 − distance`**. So a paraphrase at distance `0.14` is `0.86` relevance, and the unrelated question at distance `0.91` is `0.09` relevance. The cell below prints **both** columns so the mapping is explicit — and it's why `score_threshold` (a *maximum distance*) is a **small** number, not a large one: you're capping how far apart two prompts may be and still count as a cache hit.

In [7]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

QUESTION   = "How can semantic caching reduce LLM costs?"
PARAPHRASE = "How can I use semantic caching to save on LLM spend?"   # same meaning
UNRELATED  = "What is the weather forecast for London?"               # different meaning

def cosine_distance(a, b):
    import math
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a)); nb = math.sqrt(sum(y * y for y in b))
    return 1.0 - dot / (na * nb)          # 0.0 = identical meaning, higher = more different

vq, vp, vu = embeddings.embed_documents([QUESTION, PARAPHRASE, UNRELATED])
d_para, d_unrel = cosine_distance(vq, vp), cosine_distance(vq, vu)

print(f"embedding dimensions: {len(vq)}\n")
# Distance is what the cache thresholds on (smaller = closer). Relevance = 1 - distance
# is the intuitive 0->1 score (larger = closer) — same signal, opposite ordering.
print(f"{'pair':<12}{'distance':>12}{'relevance':>12}")
print(f"{'paraphrase':<12}{d_para:>12.3f}{1 - d_para:>12.3f}   <- hit  (same meaning)")
print(f"{'unrelated':<12}{d_unrel:>12.3f}{1 - d_unrel:>12.3f}   <- miss (different)")

embedding dimensions:        1536
paraphrase distance (hit):   0.136
unrelated distance  (miss):  0.909


The paraphrase sits far below the unrelated question, so a **`score_threshold` of `0.25`** cleanly separates them: a paraphrase (distance ≈ 0.14) is a hit; the weather question (≈ 0.9) is a miss. This is the single most important knob to tune when you adopt semantic caching — too loose and you serve wrong answers; too tight and you never get a hit.

---
# Part 5 — Create the two Oracle tables

With the model chosen, we create the cache and history tables. Both classes expose static `drop_table` / `create_tables` helpers, so we make the notebook **idempotent** by dropping first.

`OracleSemanticCache` constructor arguments:
- `client` — the live Oracle connection.
- `embedding` — the `Embeddings` object from Part 4 (this is what makes the cache *semantic*).
- `table_name` — where cache rows live.
- `distance_strategy=DistanceStrategy.COSINE` — how similarity is measured; cosine pairs naturally with normalized text embeddings.
- `create_index_if_missing=False` — for this tiny demo, exact search is fine; skip building a vector index.
- `score_threshold=0.25` — the calibrated maximum distance for a hit.

In [ ]:
from langchain_oracledb import OracleChatMessageHistory, OracleSemanticCache
from langchain_oracledb.vectorstores import DistanceStrategy

CACHE_TABLE = "LC_SEM_CACHE_DEMO"
HISTORY_TABLE = "LC_CHAT_HISTORY_DEMO"

OracleSemanticCache.drop_table(connection, CACHE_TABLE)
OracleChatMessageHistory.drop_table(connection, HISTORY_TABLE)

semantic_cache = OracleSemanticCache(
    client=connection,
    embedding=embeddings,
    table_name=CACHE_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
    create_index_if_missing=False,
    score_threshold=0.25,
)

✅ Fresh tables ready: LC_SEM_CACHE_DEMO (cache) and LC_CHAT_HISTORY_DEMO (history).


### `OracleChatMessageHistory` — the arguments that matter

`OracleChatMessageHistory` is durable, `session_id`-scoped message storage, and you touch it in two ways:

**Setup (once):** `OracleChatMessageHistory.create_tables(connection, HISTORY_TABLE)` (next cell) runs the DDL that creates the backing table — separate from any per-session object, so you provision the schema a single time.

**Per session (each request):** the constructor (used later in `get_session_history`) takes:

| Argument | What it does |
|----------|--------------|
| `session_id` | The key that scopes this transcript. Two objects with the same `session_id` see the same ordered messages; different ids are fully isolated. |
| `client` | The live Oracle connection — the *same* one the cache uses, so history and cache share a single connection. |
| `table_name` | Which table holds the rows (`HISTORY_TABLE` here). |
| `create_table` / `create_index` | Whether to run DDL when the object is constructed. We pass **`False`** for both because `create_tables` already made them, so each per-request object is cheap and never re-issues DDL. |
| `history_size` | *(optional)* caps how many recent messages are **read back**, to bound prompt/token cost. It limits reads only — it never deletes rows (we prove this later). |

In [9]:
OracleChatMessageHistory.create_tables(connection, HISTORY_TABLE)
print(f"✅ Fresh tables ready: {CACHE_TABLE} (cache) and {HISTORY_TABLE} (history).")

✅ Fresh tables ready: LC_SEM_CACHE_DEMO (cache) and LC_CHAT_HISTORY_DEMO (history).


A small helper to read row counts straight from Oracle — we'll use it to *prove* what the cache stores versus what the history stores at each step.

In [10]:
def table_row_count(table_name: str) -> int:
    with connection.cursor() as cursor:
        return cursor.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]

print("cache rows:", table_row_count(CACHE_TABLE), "| history rows:", table_row_count(HISTORY_TABLE))

cache rows: 0 | history rows: 0


---
# Part 6 — A cache-first answer function

Now the logic from the flowchart, in code. There are two ways to use `OracleSemanticCache`:

1. **Model-level** — hand it to `ChatAnthropic(cache=...)` and LangChain caches by the *entire* serialized prompt. Simple, but in a chat app the growing transcript becomes part of the key, so paraphrases rarely hit.
2. **Application-level** (what we do) — call `semantic_cache.lookup(...)` / `update(...)` ourselves, keying on **only the latest user question**. This is what lets a paraphrase hit *regardless of each session's prior history*.

`Generation(text=...)` is the LangChain container the cache stores; `lookup` returns a list of them (or `None` on a miss). The `llm_string` argument is the **model namespace** — cache entries for `support-v1` never satisfy a lookup under `support-v2`.

In [11]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.outputs import Generation
from langchain_core.runnables import RunnableConfig, RunnableLambda

# The "expensive" model whose repeated calls we want to avoid.
llm = ChatAnthropic(model=os.getenv("ANTHROPIC_MODEL", "claude-sonnet-5"), max_tokens=256)

MODEL_NAMESPACE = "claude-sonnet-5;support-v1"
metrics = {"llm_calls": 0}          # counts REAL Claude calls, so we can see caching work
cache_events: list[dict] = []

The tiny helper below pulls the **latest** `HumanMessage` out of the message list. That single string — not the whole transcript — is what becomes the cache key, which is exactly what lets a paraphrase hit no matter how much conversation history precedes it. It raises if there is no human message to key on.

In [12]:
def latest_human_question(messages: list[BaseMessage]) -> str:
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return str(message.content)
    raise ValueError("At least one HumanMessage is required.")


### The cache-first answer function

The function below is the entire caching policy in one place: take the latest question, **look it up** in the cache under the current model namespace, and either return the stored answer (a **HIT** — no model call) or call Claude, **store** the result, and return it (a **MISS**). It logs each outcome to `cache_events` and bumps `metrics["llm_calls"]` only on a miss, so later cells can *prove* caching skipped real calls.

It uses three `OracleSemanticCache` methods. Note we call them ourselves — an **application-level** cache — rather than attaching the cache to the model, so the key is the latest question alone (not the full serialized prompt):

| Method | Signature | What it does |
|--------|-----------|--------------|
| `lookup` | `lookup(prompt, llm_string)` | Embeds `prompt`, runs a vector search in Oracle, and returns the stored `[Generation]` of the nearest entry **if** its distance is within `score_threshold` — otherwise `None` (a miss). `llm_string` scopes the search so only entries from the same model/version can match. |
| `update` | `update(prompt, llm_string, return_val)` | Writes a new cache row: embeds `prompt` and stores it with `return_val` (a list of `Generation` objects) under the `llm_string` namespace. Called on a miss, right after the real model answers. |
| `clear` | `clear(prompt=…, llm_string=…)` | Deletes cache entries. With no arguments it empties the cache; with `prompt=` + `llm_string=` it removes exactly one entry (we use the targeted form near the end). |

> **Why `llm_string` (the "model namespace")?** It separates answers by *which model/config produced them*. Ask the same question under a new model version and it's a deliberate miss — you never serve an answer generated by a model you've since changed.

In [ ]:
def cache_first_answer(messages: list[BaseMessage], config: RunnableConfig) -> AIMessage:
    question = latest_human_question(messages)
    namespace = config.get("configurable", {}).get("model_namespace", MODEL_NAMESPACE)

    cached = semantic_cache.lookup(question, namespace)
    
    if cached is None:
        status = "MISS"
        metrics["llm_calls"] += 1                       # a real, billable Claude call
        answer = llm.invoke(question).content
        semantic_cache.update(question, namespace, [Generation(text=answer)])
    else:
        status = "HIT"
        answer = cached[0].text                         # reused — no model call

    event = {"status": status, "cache_key": question,
             "model_namespace": namespace, "input_message_count": len(messages)}
    cache_events.append(event)
    return AIMessage(content=answer, additional_kwargs=event)

print("Cache-first answer function ready (keys on the latest question only).")

Cache-first answer function ready (keys on the latest question only).


---
# Part 7 — Wire in durable history with `RunnableWithMessageHistory`

Continuing from Part 6, we now let LangChain load and save each session's transcript automatically. `get_session_history(session_id)` returns an `OracleChatMessageHistory` bound to that session; `RunnableWithMessageHistory` loads its messages *before* calling `cache_first_answer` and appends the new human/AI pair *after*.

> **Current LangChain note:** `RunnableWithMessageHistory` is deprecated in favor of [LangGraph persistence](https://docs.langchain.com/oss/python/langgraph/persistence). It still works and is used here because this notebook is specifically about the `OracleChatMessageHistory` primitive; we suppress only that one construction-time deprecation warning.

In [14]:
import warnings
from langchain_core.runnables.history import RunnableWithMessageHistory

def get_session_history(session_id: str) -> OracleChatMessageHistory:
    return OracleChatMessageHistory(
        session_id=session_id,
        client=connection,
        table_name=HISTORY_TABLE,
        create_table=False,
        create_index=False,
    )

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=r".*RunnableWithMessageHistory.*deprecated.*")
    chat = RunnableWithMessageHistory(RunnableLambda(cache_first_answer), get_session_history)

def ask(session_id: str, question: str, model_namespace: str = MODEL_NAMESPACE) -> AIMessage:
    return chat.invoke(
        [HumanMessage(content=question)],
        config={"configurable": {"session_id": session_id, "model_namespace": model_namespace}},
    )

print(f"RunnableWithMessageHistory is backed by {HISTORY_TABLE}.")

RunnableWithMessageHistory is backed by LC_CHAT_HISTORY_DEMO.


---
# Part 8 — Watch the cache work across two different sessions

This is the payoff. Two users:
- **`session-red`** asks the base question — a cache **miss**, so Claude is called once.
- **`session-blue`** has a *different* prior conversation, then asks a **paraphrase** — a cache **hit**, so Claude is **not** called, and blue gets red's answer.

We seed blue with unrelated history first, so its request carries extra messages. Watch that the cache key is still just the latest question (`input_message_count` is 3 for blue, but the key is the paraphrase alone).

In [15]:
RED_SESSION, BLUE_SESSION = "session-red", "session-blue"

# Blue already had a conversation before asking its question.
get_session_history(BLUE_SESSION).add_messages([
    HumanMessage(content="Remember that I prefer concise replies."),
    AIMessage(content="Noted — I'll keep replies concise."),
])

red = ask(RED_SESSION, QUESTION)          # MISS  → one real Claude call
blue = ask(BLUE_SESSION, PARAPHRASE)      # HIT   → zero Claude calls

print(f"red  [{red.additional_kwargs['status']}]  {red.content}\n")
print(f"blue [{blue.additional_kwargs['status']}] {blue.content}\n")
print(f"real Claude calls so far: {metrics['llm_calls']}   (paraphrase was served from Oracle)")
print(f"blue saw {blue.additional_kwargs['input_message_count']} messages, "
      f"but its cache key was only: {blue.additional_kwargs['cache_key']!r}")
print(f"cache rows: {table_row_count(CACHE_TABLE)} | history rows: {table_row_count(HISTORY_TABLE)}")

red  [MISS]  # Semantic Caching for LLM Cost Reduction

Semantic caching reduces LLM costs by storing and reusing responses based on the *meaning* of queries rather than exact text matches. Here's how it works and why it saves money:

## How It Works

1. **Embedding generation**: When a query comes in, it's converted into a vector embedding that captures its semantic meaning
2. **Similarity search**: This embedding is compared against cached query embeddings using vector similarity (cosine similarity, etc.)
3. **Cache hit/miss decision**: If similarity exceeds a threshold, the cached response is returned instead of calling the LLM
4. **Cache miss handling**: If no sufficiently similar match exists, the query goes to the LLM, and the new query-response pair is

blue [HIT] # Semantic Caching for LLM Cost Reduction

Semantic caching reduces LLM costs by storing and reusing responses based on the *meaning* of queries rather than exact text matches. Here's how it works and why it saves mone

### Persistence + isolation: reopen the transcripts

New `OracleChatMessageHistory` handles simulate a later request or a process restart. Each session recovers its **own** ordered rows — red never sees blue's paraphrase, and blue never sees red's question. Same cache, fully separate transcripts.

In [16]:
red_msgs = get_session_history(RED_SESSION).messages
blue_msgs = get_session_history(BLUE_SESSION).messages

print("RED transcript:")
for m in red_msgs:
    print(f"  {type(m).__name__:12s} {m.content[:70]}")
print("\nBLUE transcript:")
for m in blue_msgs:
    print(f"  {type(m).__name__:12s} {m.content[:70]}")
print(f"\nParaphrase leaked into red? {PARAPHRASE in [m.content for m in red_msgs]}")
print(f"Base question leaked into blue? {QUESTION in [m.content for m in blue_msgs]}")

RED transcript:
  HumanMessage How can semantic caching reduce LLM costs?
  AIMessage    # Semantic Caching for LLM Cost Reduction

Semantic caching reduces LL

BLUE transcript:
  HumanMessage Remember that I prefer concise replies.
  AIMessage    Noted — I'll keep replies concise.
  HumanMessage How can I use semantic caching to save on LLM spend?
  AIMessage    # Semantic Caching for LLM Cost Reduction

Semantic caching reduces LL

Paraphrase leaked into red? False
Base question leaked into blue? False


### `history_size` bounds reads, not storage

Add two more turns to red, then open a reader with `history_size=4`. It returns only the last four messages, while Oracle still holds all six — `history_size` limits what you *read back into the prompt* (to control context/token cost), it never deletes rows.

In [17]:
get_session_history(RED_SESSION).add_messages([
    HumanMessage(content="What persists the transcript?"),
    AIMessage(content="OracleChatMessageHistory persists it in Oracle."),
    HumanMessage(content="Does a bounded read delete older rows?"),
    AIMessage(content="No — history_size limits reads; it never deletes rows."),
])

full_red = get_session_history(RED_SESSION)
bounded_red = OracleChatMessageHistory(
    session_id=RED_SESSION, client=connection, table_name=HISTORY_TABLE,
    create_table=False, create_index=False, history_size=4,
)
print(f"unbounded reader sees {len(full_red.messages)} messages")
print(f"history_size=4 reader sees {len(bounded_red.messages)} messages")
with connection.cursor() as cur:
    stored = cur.execute(f'SELECT COUNT(*) FROM "{HISTORY_TABLE}" WHERE "session_id" = :s',
                         {"s": RED_SESSION}).fetchone()[0]
print(f"Oracle still stores {stored} red rows (bounded read deleted nothing).")

unbounded reader sees 6 messages
history_size=4 reader sees 4 messages
Oracle still stores 6 red rows (bounded read deleted nothing).


### A new model namespace misses; an unrelated question misses

Two more ways to get a **miss**, both important for correctness:
- Ask the **exact same** question under a different `model_namespace` (`support-v2`) — cache entries are namespaced by model, so upgrading the model doesn't serve stale answers.
- Ask the **unrelated** weather question — its vector is far outside `score_threshold`.

Each miss triggers one real Claude call, so `llm_calls` climbs by two.

In [18]:
GREEN_SESSION = "session-green"
calls_before = metrics["llm_calls"]

ns_v2 = ask(GREEN_SESSION, QUESTION, model_namespace="claude-sonnet-5;support-v2")  # MISS (new namespace)
weather = ask(BLUE_SESSION, UNRELATED)                                              # MISS (far vector)

print(f"same question, new namespace → {ns_v2.additional_kwargs['status']}")
print(f"unrelated weather question   → {weather.additional_kwargs['status']}")
print(f"real Claude calls: {calls_before} → {metrics['llm_calls']} (+2 misses)")
print(f"cache rows now: {table_row_count(CACHE_TABLE)}")

same question, new namespace → MISS
unrelated weather question   → MISS
real Claude calls: 1 → 3 (+2 misses)
cache rows now: 3


### Inspect and selectively clear one entry

The cache now holds three rows: the base question under `support-v1`, the same question under `support-v2`, and the weather question under `support-v1`. `clear(prompt=..., llm_string=...)` removes exactly one — here the `support-v2` entry — leaving the two `support-v1` entries (including the paraphrase-hittable one) intact, and never touching the chat transcript.

In [19]:
print("before:", table_row_count(CACHE_TABLE), "cache rows")
semantic_cache.clear(prompt=QUESTION, llm_string="claude-sonnet-5;support-v2")
print("after :", table_row_count(CACHE_TABLE), "cache rows")

print("Q under support-v2 now:", "present" if semantic_cache.lookup(QUESTION, "claude-sonnet-5;support-v2") else "gone")
print("paraphrase under support-v1 still hits:", semantic_cache.lookup(PARAPHRASE, MODEL_NAMESPACE) is not None)
print("history rows untouched:", table_row_count(HISTORY_TABLE))

before: 3 cache rows
after : 2 cache rows
Q under support-v2 now: gone
paraphrase under support-v1 still hits: True
history rows untouched: 14


---
# Part 9 — Clean up

Drop only the two demo tables and close the connection. In a real application you'd provision these tables once, reuse a connection pool, and apply a retention policy instead of dropping at shutdown.

In [20]:
OracleSemanticCache.drop_table(connection, CACHE_TABLE)
OracleChatMessageHistory.drop_table(connection, HISTORY_TABLE)
connection.close()
print("✅ Dropped both demo tables and closed the Oracle connection.")

✅ Dropped both demo tables and closed the Oracle connection.


---
# Recap

You built a cache-first support assistant on the **PALO** stack with an OpenAI vectorizer:

- **`OracleSemanticCache`** turned "same meaning" into "small cosine distance," so a **paraphrase hit the cache across a different session** and skipped a billable Claude call — after you *calibrated* `score_threshold` to the embedding model.
- **`OracleChatMessageHistory`** kept every session's transcript durable and isolated, and `history_size` bounded reads without deleting rows.
- The two scopes stayed independent: **cache key = latest question + model namespace**, **transcript = session_id**.

**Where to go next:** enable a real ANN index (`create_index_if_missing=True`) for large caches; move to a connection pool; and for full agent memory (checkpoints + cross-thread stores) see [LangGraph persistence on Oracle](https://docs.langchain.com/oss/python/langgraph/persistence). Reference docs: [langchain-oracledb](https://pypi.org/project/langchain-oracledb/) · [OracleVS vectorstore](https://docs.langchain.com/oss/python/integrations/vectorstores/oracle) · [OpenAIEmbeddings](https://docs.langchain.com/oss/python/integrations/text_embedding/openai) · [ChatAnthropic](https://docs.langchain.com/oss/python/integrations/chat/anthropic) · [Oracle AI Vector Search](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/overview-ai-vector-search.html).